# DESI Legacy re-import



In [ ]:
from dask.distributed import Client
from hats_import.pipeline import pipeline_with_client, pipeline
from hats_import import ImportArguments, CollectionArguments, VerificationArguments
import hats_import

hats_import.__version__

In [ ]:
# !mkdir /epyc/data3/hats/catalogs/legacysurvey/legacysurvey_collection

In [ ]:
args = ImportArguments.reimport_from_hats(
    "/astro/store/shire/hats/catalogs/legacysurvey_dr10.1/legacysurvey_dr10.1",
    "/astro/store/shire/hats/staging/legacysurvey/legacysurvey_collection/",
    output_artifact_name="legacysurvey",

    pixel_threshold=500_000,
    highest_healpix_order=10,
    skymap_alt_orders=[2, 4, 6, 8],
    row_group_kwargs={"num_rows": 100_000},

    completion_email_address="delucchi@andrew.cmu.edu",
    progress_bar=True,
    simple_progress_bar=True,
)

with Client(local_directory="/astro/store/shire/hats/tmp", n_workers=20, threads_per_worker=1) as client:
    pipeline_with_client(args, client)

In [ ]:
args = (
    CollectionArguments(
        output_artifact_name="legacysurvey_collection",
        output_path="/astro/store/shire/hats/staging/legacysurvey/",
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(
        output_artifact_name="legacysurvey",
    )
    .add_margin(margin_threshold=10.0, is_default=True)
    .add_index(indexing_column="OBJID", include_healpix_29=False)
)

with Client(local_directory="/astro/store/shire/hats/tmp", n_workers=10, threads_per_worker=1) as client:
    pipeline_with_client(args, client)

In [ ]:
args = VerificationArguments(
    input_catalog_path="/astro/store/shire/hats/staging/legacysurvey/legacysurvey_collection",
    output_path="./verification/legacysurvey",
)
pipeline(args)